In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")
REPORT_DIR = Path("../reports")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
transactions = pd.read_csv(DATA_DIR / "customer_transaction_data.csv")
campaigns = pd.read_csv(DATA_DIR / "campaign_data.csv")
customers = pd.read_csv(DATA_DIR / "customer_demographics.csv")

print("Datasets loaded successfully!")
print("-" * 45)

print("Training data:", train.shape)
print("Transaction data:", transactions.shape)
print("Campaign data:", campaigns.shape)
print("Customer demographics:", customers.shape)

Datasets loaded successfully!
---------------------------------------------
Training data: (78369, 5)
Transaction data: (1324566, 7)
Campaign data: (28, 4)
Customer demographics: (760, 7)


In [2]:
# Display columns from the training dataset
print("TRAINING DATA COLUMNS")
print("-" * 50)
print(train.columns.tolist())

# Display columns from the transaction dataset
print("\nTRANSACTION DATA COLUMNS")
print("-" * 50)
print(transactions.columns.tolist())

# Display columns from the campaign dataset
print("\nCAMPAIGN DATA COLUMNS")
print("-" * 50)
print(campaigns.columns.tolist())

# Display columns from the customer dataset
print("\nCUSTOMER DATA COLUMNS")
print("-" * 50)
print(customers.columns.tolist())

TRAINING DATA COLUMNS
--------------------------------------------------
['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status']

TRANSACTION DATA COLUMNS
--------------------------------------------------
['date', 'customer_id', 'item_id', 'quantity', 'selling_price', 'other_discount', 'coupon_discount']

CAMPAIGN DATA COLUMNS
--------------------------------------------------
['campaign_id', 'campaign_type', 'start_date', 'end_date']

CUSTOMER DATA COLUMNS
--------------------------------------------------
['customer_id', 'age_range', 'marital_status', 'rented', 'family_size', 'no_of_children', 'income_bracket']


In [3]:
# Find discount-related columns
discount_cols = [
    col for col in transactions.columns
    if "discount" in col.lower()
]

print("Discount-related columns:")
print(discount_cols)

print("\nDiscount variable summary:")
if discount_cols:
    display(transactions[discount_cols].describe().T)
else:
    print("No discount-related columns found.")

print("\nSample transaction records:")
display(transactions.head())

print("\nTraining outcome distribution:")
display(
    train["redemption_status"]
    .value_counts()
    .rename_axis("redemption_status")
    .reset_index(name="count")
)

Discount-related columns:
['other_discount', 'coupon_discount']

Discount variable summary:


,count,mean,std,min,25%,50%,75%,max
other_discount,1324566.0,-17.768707,37.888674,-3120.31,-23.15,-1.78,0.0,0.0
coupon_discount,1324566.0,-0.594898,7.069367,-1992.23,0.00,0.00,0.0,0.0



Sample transaction records:


,date,customer_id,item_id,quantity,selling_price,other_discount,coupon_discount
0,2012-01-02,1501,26830,1,35.26,-10.69,0.0
1,2012-01-02,1501,54253,1,53.43,-13.89,0.0
2,2012-01-02,1501,31962,1,106.50,-14.25,0.0
3,2012-01-02,1501,33647,1,67.32,0.00,0.0
4,2012-01-02,1501,48199,1,71.24,-28.14,0.0



Training outcome distribution:


,redemption_status,count
0,0,77640
1,1,729


In [4]:
# 1. Check whether redemption_status exists in each dataset
print("OUTCOME AVAILABILITY")
print("-" * 50)

for name, df in {
    "Training": train,
    "Transactions": transactions,
    "Campaigns": campaigns,
    "Customers": customers
}.items():
    print(
        f"{name}: "
        f"{'redemption_status' in df.columns}"
    )


# 2. Check potential treatment-related columns
print("\nPOTENTIAL TREATMENT VARIABLES")
print("-" * 50)

for name, df in {
    "Training": train,
    "Transactions": transactions,
    "Campaigns": campaigns
}.items():
    candidates = [
        col for col in df.columns
        if any(
            word in col.lower()
            for word in [
                "discount",
                "campaign",
                "coupon",
                "price",
                "offer"
            ]
        )
    ]

    print(f"\n{name}:")
    print(candidates)


# 3. Check campaign IDs and coupon IDs in the training data
print("\nTRAINING DATA IDENTIFIERS")
print("-" * 50)

for col in ["campaign_id", "coupon_id", "customer_id"]:
    if col in train.columns:
        print(
            f"{col}: "
            f"{train[col].nunique()} unique values"
        )

OUTCOME AVAILABILITY
--------------------------------------------------
Training: True
Transactions: False
Campaigns: False
Customers: False

POTENTIAL TREATMENT VARIABLES
--------------------------------------------------

Training:
['campaign_id', 'coupon_id']

Transactions:
['selling_price', 'other_discount', 'coupon_discount']

Campaigns:
['campaign_id', 'campaign_type']

TRAINING DATA IDENTIFIERS
--------------------------------------------------
campaign_id: 18 unique values
coupon_id: 866 unique values
customer_id: 1428 unique values


In [5]:
# Display campaign dataset information
print("CAMPAIGN DATASET")
print("-" * 50)

print("Shape:", campaigns.shape)
print("\nColumns:")
print(campaigns.columns.tolist())

print("\nFirst 10 campaign records:")
display(campaigns.head(10))

print("\nUnique campaign IDs:")
print(campaigns["campaign_id"].nunique())

# Summarize categorical campaign variables
categorical_cols = campaigns.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    print(f"\nValue counts for {col}:")
    display(campaigns[col].value_counts(dropna=False))

CAMPAIGN DATASET
--------------------------------------------------
Shape: (28, 4)

Columns:
['campaign_id', 'campaign_type', 'start_date', 'end_date']

First 10 campaign records:


,campaign_id,campaign_type,start_date,end_date
0,24,Y,21/10/13,20/12/13
1,25,Y,21/10/13,22/11/13
2,20,Y,07/09/13,16/11/13
3,23,Y,08/10/13,15/11/13
4,21,Y,16/09/13,18/10/13
5,22,X,16/09/13,18/10/13
6,18,X,10/08/13,04/10/13
7,19,Y,26/08/13,27/09/13
8,17,Y,29/07/13,30/08/13
9,16,Y,15/07/13,16/08/13



Unique campaign IDs:
28

Value counts for campaign_type:


C:\Users\KESHAV DEV GAUR\AppData\Local\Temp\ipykernel_29584\1364252616.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = campaigns.select_dtypes(


campaign_type
Y    22
X     6
Name: count, dtype: int64


Value counts for start_date:


start_date
21/10/13    2
16/09/13    2
22/04/13    2
07/09/13    1
08/10/13    1
10/08/13    1
26/08/13    1
29/07/13    1
15/07/13    1
19/05/13    1
08/04/13    1
11/03/13    1
16/02/13    1
02/02/13    1
28/01/13    1
22/12/12    1
12/01/13    1
07/01/13    1
12/12/12    1
17/12/12    1
19/11/12    1
08/10/12    1
16/09/12    1
25/08/12    1
12/08/12    1
Name: count, dtype: int64


Value counts for end_date:


end_date
18/10/13    2
18/01/13    2
20/12/13    1
22/11/13    1
16/11/13    1
15/11/13    1
04/10/13    1
27/09/13    1
30/08/13    1
16/08/13    1
05/07/13    1
07/06/13    1
24/05/13    1
10/05/13    1
12/04/13    1
05/04/13    1
08/03/13    1
01/03/13    1
16/02/13    1
15/02/13    1
08/02/13    1
04/01/13    1
30/11/12    1
16/11/12    1
27/10/12    1
21/09/12    1
Name: count, dtype: int64

In [6]:
# Preliminary causal variable assessment

causal_assessment = pd.DataFrame([
    {
        "Variable": "redemption_status",
        "Role": "Outcome",
        "Dataset": "train.csv",
        "Current Understanding": "Observed coupon redemption outcome",
        "Status": "Identified"
    },
    {
        "Variable": "coupon_discount",
        "Role": "Potential treatment",
        "Dataset": "customer_transaction_data.csv",
        "Current Understanding": "Recorded transaction discount",
        "Status": "Treatment assignment not established"
    },
    {
        "Variable": "other_discount",
        "Role": "Potential treatment",
        "Dataset": "customer_transaction_data.csv",
        "Current Understanding": "Recorded other discount",
        "Status": "Treatment assignment not established"
    },
    {
        "Variable": "campaign_id",
        "Role": "Potential exposure indicator",
        "Dataset": "train.csv",
        "Current Understanding": "Identifies the associated campaign",
        "Status": "Assignment mechanism requires investigation"
    },
    {
        "Variable": "customer_id",
        "Role": "Identifier",
        "Dataset": "Multiple datasets",
        "Current Understanding": "Connects records for a customer",
        "Status": "Not a treatment by itself"
    },
    {
        "Variable": "customer demographics",
        "Role": "Potential covariates",
        "Dataset": "customer_demographics.csv",
        "Current Understanding": "Describes customer characteristics",
        "Status": "Missingness and timing require assessment"
    }
])

display(causal_assessment)

,Variable,Role,Dataset,Current Understanding,Status
0,redemption_status,Outcome,train.csv,Observed coupon redemption outcome,Identified
1,coupon_discount,Potential treatment,customer_transaction_data.csv,Recorded transaction discount,Treatment assignment not established
2,other_discount,Potential treatment,customer_transaction_data.csv,Recorded other discount,Treatment assignment not established
3,campaign_id,Potential exposure indicator,train.csv,Identifies the associated campaign,Assignment mechanism requires investigation
4,customer_id,Identifier,Multiple datasets,Connects records for a customer,Not a treatment by itself
5,customer demographics,Potential covariates,customer_demographics.csv,Describes customer characteristics,Missingness and timing require assessment
